# Zerobus Ingest Benchmark Driver

Run all four ingest patterns (gRPC sync, gRPC async, HTTP sync, HTTP async) in a loop
using `zbhelper.ingest_benchmark`.  Each iteration ingests `_N` rows and prints the
full Step-4d metrics block.

**Prerequisites:** Step 2 cells from any of the four demo notebooks must run first so
`SERVER_ENDPOINT`, `DATABRICKS_WORKSPACE_URL`, `DATABRICKS_WORKSPACE_ID`,
`ZEROBUS_INGEST_URL`, `CLIENT_ID`, `CLIENT_SECRET`, `TABLE_NAME`, `CATALOG`, `SCHEMA`
are all defined in this kernel.

### Step 1: Install dependencies

In [1]:
%pip install --quiet databricks-zerobus-ingest-sdk aiohttp requests


Note: you may need to restart the kernel to use updated packages.


### Step 2: Import benchmark module

In [2]:
%load_ext autoreload
%autoreload 2

import zbhelper.ingest_benchmark as zb

print("zbhelper.ingest_benchmark loaded")


ModuleNotFoundError: No module named 'zbhelper'

### Step 3: Configuration

All variables below must already be defined — run the Step 2 cells from any demo
notebook first, or assign them directly here.

```
SERVER_ENDPOINT           # gRPC endpoint  (from Step 2.a)
DATABRICKS_WORKSPACE_URL  # workspace URL  (from Step 2.a)
DATABRICKS_WORKSPACE_ID   # numeric ID     (from Step 2.a)
ZEROBUS_INGEST_URL        # HTTP base URL  (same as SERVER_ENDPOINT)
CLIENT_ID                 # SP app ID      (from Step 2.c)
CLIENT_SECRET             # SP secret      (from Step 2.c)
TABLE_NAME                # catalog.schema.table (from Step 2.d)
CATALOG / SCHEMA          # UC components  (from Step 2.d)
```

In [ ]:
_N       = 1000          # total rows per iteration
_SINGLES = min(10, _N)  # single-row phase (4a) row count
_ITERS   = 1             # number of full benchmark loops

# Set False to skip a pattern
_RUN_GRPC_SYNC  = True
_RUN_GRPC_ASYNC = True
_RUN_HTTP_SYNC  = True
_RUN_HTTP_ASYNC = True


### Step 4: Benchmark loop

In [ ]:
for _iteration in range(1, _ITERS + 1):
    if _ITERS > 1:
        print(f"\n{'='*60}")
        print(f"Iteration {_iteration} / {_ITERS}")
        print(f"{'='*60}")

    records_4a = zb.build_records(_SINGLES)
    records_4b = zb.build_records(_N - _SINGLES, offset=_SINGLES)

    # ------------------------------------------------------------------ #
    # gRPC sync                                                            #
    # ------------------------------------------------------------------ #
    if _RUN_GRPC_SYNC:
        print("\n--- gRPC sync ---")
        baseline = zb.fetch_row_baseline(spark, TABLE_NAME)
        stream, stream_open_s = zb.open_grpc_stream_sync(
            SERVER_ENDPOINT, DATABRICKS_WORKSPACE_URL,
            CLIENT_ID, CLIENT_SECRET, TABLE_NAME,
        )
        singles = zb.ingest_singles_grpc_sync(stream, records_4a)
        batch   = zb.ingest_batch_and_close_grpc_sync(stream, records_4b, stream_open_s=stream_open_s)
        vis     = zb.poll_visibility(
            spark, TABLE_NAME, baseline["count"] + _N,
            batch.t_after_close, singles.t_4a0,
        )
        zb.print_metrics(TABLE_NAME, _N, baseline["count"], baseline, singles, batch, vis)

    # ------------------------------------------------------------------ #
    # gRPC async                                                           #
    # ------------------------------------------------------------------ #
    if _RUN_GRPC_ASYNC:
        print("\n--- gRPC async ---")
        baseline = zb.fetch_row_baseline(spark, TABLE_NAME)
        stream, stream_open_s = await zb.open_grpc_stream_async(
            SERVER_ENDPOINT, DATABRICKS_WORKSPACE_URL,
            CLIENT_ID, CLIENT_SECRET, TABLE_NAME,
        )
        singles = await zb.ingest_singles_grpc_async(stream, records_4a)
        batch   = await zb.ingest_batch_and_close_grpc_async(stream, records_4b, stream_open_s=stream_open_s)
        vis     = zb.poll_visibility(
            spark, TABLE_NAME, baseline["count"] + _N,
            batch.t_after_close, singles.t_4a0,
        )
        zb.print_metrics(TABLE_NAME, _N, baseline["count"], baseline, singles, batch, vis)

    # ------------------------------------------------------------------ #
    # HTTP sync                                                            #
    # ------------------------------------------------------------------ #
    if _RUN_HTTP_SYNC:
        import requests as _requests
        print("\n--- HTTP sync ---")
        baseline = zb.fetch_row_baseline(spark, TABLE_NAME)
        token      = zb.fetch_http_token(
            DATABRICKS_WORKSPACE_URL, DATABRICKS_WORKSPACE_ID,
            CLIENT_ID, CLIENT_SECRET, CATALOG, SCHEMA, TABLE_NAME,
        )
        insert_url = zb.http_insert_url(ZEROBUS_INGEST_URL, TABLE_NAME)
        with _requests.Session() as _http_session:
            singles = zb.ingest_singles_http_sync(insert_url, token, records_4a, session=_http_session)
            batch   = zb.ingest_batch_http_sync(insert_url, token, records_4b, session=_http_session)
        vis = zb.poll_visibility(
            spark, TABLE_NAME, baseline["count"] + _N,
            batch.t_after_close, singles.t_4a0,
        )
        zb.print_metrics(TABLE_NAME, _N, baseline["count"], baseline, singles, batch, vis)

    # ------------------------------------------------------------------ #
    # HTTP async                                                           #
    # ------------------------------------------------------------------ #
    if _RUN_HTTP_ASYNC:
        import aiohttp as _aiohttp
        print("\n--- HTTP async ---")
        baseline = zb.fetch_row_baseline(spark, TABLE_NAME)
        token      = zb.fetch_http_token(
            DATABRICKS_WORKSPACE_URL, DATABRICKS_WORKSPACE_ID,
            CLIENT_ID, CLIENT_SECRET, CATALOG, SCHEMA, TABLE_NAME,
        )
        insert_url = zb.http_insert_url(ZEROBUS_INGEST_URL, TABLE_NAME)
        async with _aiohttp.ClientSession() as _http_session:
            singles = await zb.ingest_singles_http_async(insert_url, token, records_4a, session=_http_session)
            batch   = await zb.ingest_batch_http_async(insert_url, token, records_4b, session=_http_session)
        vis = zb.poll_visibility(
            spark, TABLE_NAME, baseline["count"] + _N,
            batch.t_after_close, singles.t_4a0,
        )
        zb.print_metrics(TABLE_NAME, _N, baseline["count"], baseline, singles, batch, vis)
